# Notebook 21: LangGraph - Integration with Tools & APIs

**🎯 Goal:** To learn how to empower LangGraph agents by giving them access to tools and external APIs, and how to stream their outputs for a responsive user experience.

## 🧩 Concept Overview

An agent's true power comes from its ability to interact with the outside world. This is done by giving the agent **tools**.

- **Tool Calling:** Modern LLMs can be 'bound' to a set of tools. When you ask the LLM a question, it can decide if it needs to use a tool to answer it and will output a special 'tool call' message.
- **Custom Tools:** You can easily define your own tools for any purpose, from searching a database to calling a specific API.
- **Streaming:** For a better user experience, you can stream the output of your agent in real-time, showing the user what the agent is 'thinking' token-by-token.

### 6.1. Tool Calling with LLMs

The standard agent loop in LangGraph for tool calling is:
1.  **Agent Node:** Call the LLM. The LLM's response might be a final answer or a request to call a tool.
2.  **Conditional Edge:** Check if the LLM requested a tool call.
3.  **Action Node:** If a tool was requested, execute it and get the result.
4.  **Loop:** Go back to the Agent Node, providing the tool's result so the LLM can formulate a final answer.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import BaseMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model="gpt-4o")

# Define a custom tool
@tool
def search_weather(city: str) -> str:
    """Searches for the weather in a given city."""
    print(f"--- Searching weather for {city} ---")
    if city.lower() == "san francisco":
        return "It's 65°F and sunny in San Francisco."
    return "Sorry, I don't know the weather for that city."

class ToolState(TypedDict):
    messages: Annotated[list, operator.add]

# Bind the tool to the LLM
tools = [search_weather]
llm_with_tools = llm.bind_tools(tools)
tool_executor = ToolExecutor(tools)

# Agent node
def agent(state: ToolState):
    response = llm_with_tools.invoke(state['messages'])
    return {"messages": [response]}

# Action node
def action(state: ToolState):
    tool_calls = state['messages'][-1].tool_calls
    tool_outputs = tool_executor.batch(tool_calls)
    tool_messages = [ToolMessage(content=str(output), tool_call_id=tc['id']) for tc, output in zip(tool_calls, tool_outputs)]
    return {"messages": tool_messages}

# Conditional edge
def router(state: ToolState):
    if state['messages'][-1].tool_calls:
        return "action"
    return "end"

workflow = StateGraph(ToolState)
workflow.add_node("agent", agent)
workflow.add_node("action", action)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", router, {"action": "action", "end": END})
workflow.add_edge('action', 'agent') # Loop back after action

app = workflow.compile()


In [ ]:
from langchain_core.messages import HumanMessage

inputs = {"messages": [HumanMessage(content="What's the weather in San Francisco?")]}
result = app.invoke(inputs)

print("\n--- Final Response ---")
print(result['messages'][-1].content)

### 6.2. Custom Tools

The `@tool` decorator is the easiest way to create a custom tool. LangChain automatically infers the schema (arguments) from the function's type hints and docstring.

You can create tools for anything: web search, code execution, database queries, file I/O, etc.

In [ ]:
@tool
def python_repl(code: str) -> str:
    """Executes a snippet of Python code."""
    try:
        result = eval(code)
        return f"Successfully executed. Result: {result}"
    except Exception as e:
        return f"Error: {e}"

# You can add this tool to your `tools` list and the LLM can use it.
print(python_repl.name)
print(python_repl.description)
print(python_repl.args)

### 6.3. Streaming Output

Instead of waiting for the final result, you can stream the output of your graph as it's being generated. This is essential for user-facing applications.

You can use `astream` or `astream_events` to get real-time updates from your compiled graph.

In [ ]:
import asyncio

async def run_streaming_example():
    inputs = {"messages": [HumanMessage(content="What's the weather in San Francisco?")]}
    async for chunk in app.astream(inputs):
        print("--- Stream Chunk ---")
        print(chunk)
        print("\n")

# In a Jupyter environment, you might need to run this with await
# asyncio.run(run_streaming_example())
print("Streaming example ready. Uncomment the line above to run.")

## 🏁 Summary

You've learned how to make your agents significantly more powerful and user-friendly.

1.  **Tools are the Key to Action:** The agent-tool-executor loop is the fundamental pattern for building agents that can interact with the world.
2.  **`@tool` Makes it Easy:** The `@tool` decorator is a simple and powerful way to define custom actions for your agents.
3.  **Streaming Improves UX:** Use `astream` to provide real-time feedback to your users, showing them the agent's progress and thoughts as they happen.

Next, we'll look at how to make our agents more resilient by handling errors gracefully.